# 02 — Dataset Splitting

**Pipeline position:** Step 2 of 7

**Purpose:** Partition the imputed dataset into three non-overlapping splits
using temporal holdout (2019 as external validation) and stratified random
9:1 split for training and internal validation.

**License:** MIT

## Prerequisites
- Run `01_data_preprocessing.ipynb` first
- Input: `./Data/imputed_data.csv`

## Outputs
- `data_training.csv` (≈360,363 records)
- `data_internal_validation.csv` (≈40,041 records)
- `data_temporal_validation.csv` (≈1,822 records)
- `data_with_splits.csv` (combined with `Dataset` label column)


In [1]:
# SPDX-License-Identifier: MIT
# ============================================================
# Dependency check — run this cell first
# ============================================================
import importlib.util

_required = ['pandas', 'numpy', 'sklearn']
_missing = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    raise ImportError(
        f"Missing packages: {_missing}\n"
        f"Install with: pip install {' '.join(_missing)}"
    )
print("Core dependencies satisfied.")

Core dependencies satisfied.


In [2]:
# ============================================================
# Configuration
# ============================================================
OUTCOME_VAR  = "SpontAbortion"
INDEX_VAR    = "Index"
DATE_VAR     = "BaseInfoDate"
RANDOM_STATE = 42           # standardised from 2026
TRAIN_RATIO  = 0.9          # 90% training / 10% internal validation
FILE_PATH    = "./Data/imputed_data.csv"


In [3]:
"""
Dataset Splitting Script
========================
Splitting strategy:
  1. Temporal validation set: 2019 data → independent external validation
  2. Training + Internal Validation: 2014-2018 data → 9:1 stratified random split

Stratification variable: SpontAbortion (outcome), ensuring consistent
outcome prevalence across all splits.

Dependencies:
  pip install pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 1. Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"       # Date column used to extract year
RANDOM_STATE = 42
TRAIN_RATIO = 0.9               # Fraction of 2014-2018 data used for training (remainder → internal validation)

# ============================================================
# 2. Load data
# ============================================================
# If using Boruta-selected features, use data_selected_features.csv
# If using all imputed variables, use imputed_data.csv
FILE_PATH = "./Data/imputed_data.csv"  # <-- Update to your actual file path

print("Loading data...")
if FILE_PATH.endswith(".csv"):
    df = pd.read_csv(FILE_PATH)
else:
    df = pd.read_excel(FILE_PATH)
print(f"Data shape: {df.shape}")

# ============================================================
# 3. Parse year from date
# ============================================================
# BaseInfoDate may be formatted as "2014/7/7" or "2014-07-07"
df[DATE_VAR] = pd.to_datetime(df[DATE_VAR], errors="coerce")
df["Year"] = df[DATE_VAR].dt.year

print(f"\nSample count by year:")
year_counts = df["Year"].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  {int(year)}: {count:,}")

# Check for records with missing year
n_missing_year = df["Year"].isnull().sum()
if n_missing_year > 0:
    print(f"\nWarning: {n_missing_year} records have missing dates — grouped with 2014-2018")
    # Records with missing dates are grouped with 2014-2018 (conservative)
    # Alternatively, exclude them: df = df.dropna(subset=["Year"])

# ============================================================
# 4. Split out temporal validation set (year 2019)
# ============================================================
mask_temporal = df["Year"] == 2019
mask_development = df["Year"].between(2014, 2018) | df["Year"].isnull()

df_temporal = df[mask_temporal].copy()
df_development = df[mask_development].copy()

print(f"\n{'='*60}")
print("Step 1: Temporal split by year")
print(f"{'='*60}")
print(f"  2014-2018 development set: {len(df_development):,}")
print(f"  2019 temporal validation:  {len(df_temporal):,}")

# ============================================================
# 5. Development set → Training + Internal Validation (9:1 stratified split)
# ============================================================
# Encode outcome variable
for subset in [df_development, df_temporal]:
    if subset[OUTCOME_VAR].dtype == object or subset[OUTCOME_VAR].dtype == bool:
        subset[OUTCOME_VAR] = subset[OUTCOME_VAR].map({
            True: 1, False: 0, "True": 1, "False": 0,
            "TRUE": 1, "FALSE": 0
        })

y_dev = df_development[OUTCOME_VAR].astype(int)

df_train, df_internal_val = train_test_split(
    df_development,
    test_size=1 - TRAIN_RATIO,    # 10% held out as internal validation
    stratify=y_dev,                # Stratify by outcome
    random_state=RANDOM_STATE,
)

print(f"\n{'='*60}")
print("Step 2: Stratified 9:1 split of development set")
print(f"{'='*60}")
print(f"  Training set          : {len(df_train):,}")
print(f"  Internal validation   : {len(df_internal_val):,}")
print(f"  Temporal validation   : {len(df_temporal):,}")

# ============================================================
# 6. Verify outcome distribution across splits
# ============================================================
def outcome_summary(data, name):
    y = data[OUTCOME_VAR].astype(int)
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    pct = y.mean() * 100
    return {
        "Dataset": name,
        "Total": len(y),
        "Positive (SA)": n_pos,
        "Negative": n_neg,
        "Positive Rate (%)": round(pct, 2),
    }

summary = pd.DataFrame([
    outcome_summary(df_train, "Training"),
    outcome_summary(df_internal_val, "Internal Validation"),
    outcome_summary(df_temporal, "Temporal Validation"),
])

print(f"\n{'='*60}")
print("Outcome distribution across splits (stratification check)")
print(f"{'='*60}")
print(summary.to_string(index=False))

# ============================================================
# 7. Assign dataset labels and save
# ============================================================
df_train["Dataset"] = "Training"
df_internal_val["Dataset"] = "Internal_Validation"
df_temporal["Dataset"] = "Temporal_Validation"

# Save combined dataset (Dataset label retained for downstream grouping)
df_all = pd.concat([df_train, df_internal_val, df_temporal], ignore_index=True)

# Drop auxiliary columns
df_all = df_all.drop(columns=["Year"], errors="ignore")

df_all.to_csv("data_with_splits.csv", index=False)
print(f"\nCombined data (with Dataset label) saved to: data_with_splits.csv")

# Save each split separately
df_train.drop(columns=["Year"], errors="ignore").to_csv(
    "data_training.csv", index=False
)
df_internal_val.drop(columns=["Year"], errors="ignore").to_csv(
    "data_internal_validation.csv", index=False
)
df_temporal.drop(columns=["Year"], errors="ignore").to_csv(
    "data_temporal_validation.csv", index=False
)

print(f"\nIndividual files saved:")
print(f"  data_training.csv              ({len(df_train):,} records)")
print(f"  data_internal_validation.csv   ({len(df_internal_val):,} records)")
print(f"  data_temporal_validation.csv   ({len(df_temporal):,} records)")

# ============================================================
# 8. Generate Table 1 framework
# ============================================================
print(f"\n{'='*60}")
print("Table 1 summary")
print(f"{'='*60}")

table1_data = {
    "": ["Training Set", "Internal Validation Set", "Temporal Validation Set"],
    "Period": ["2014–2018", "2014–2018", "2019"],
    "N": [f"{len(df_train):,}", f"{len(df_internal_val):,}", f"{len(df_temporal):,}"],
    "SA Events": [
        f"{df_train[OUTCOME_VAR].astype(int).sum():,}",
        f"{df_internal_val[OUTCOME_VAR].astype(int).sum():,}",
        f"{df_temporal[OUTCOME_VAR].astype(int).sum():,}",
    ],
    "SA Rate (%)": [
        f"{df_train[OUTCOME_VAR].astype(int).mean()*100:.2f}",
        f"{df_internal_val[OUTCOME_VAR].astype(int).mean()*100:.2f}",
        f"{df_temporal[OUTCOME_VAR].astype(int).mean()*100:.2f}",
    ],
}

table1 = pd.DataFrame(table1_data)
print(table1.to_string(index=False))

print(f"\nScript completed.")

Loading data...
Data shape: (402226, 137)

Sample count by year:
  2014: 96,875
  2015: 98,123
  2016: 128,099
  2017: 46,140
  2018: 31,167
  2019: 1,822

Step 1: Temporal split by year
  2014-2018 development set: 400,404
  2019 temporal validation:  1,822

Step 2: Stratified 9:1 split of development set
  Training set          : 360,363
  Internal validation   : 40,041
  Temporal validation   : 1,822

Outcome distribution across splits (stratification check)
            Dataset  Total  Positive (SA)  Negative  Positive Rate (%)
           Training 360363          11753    348610               3.26
Internal Validation  40041           1306     38735               3.26
Temporal Validation   1822            262      1560              14.38

Combined data (with Dataset label) saved to: data_with_splits.csv

Individual files saved:
  data_training.csv              (360,363 records)
  data_internal_validation.csv   (40,041 records)
  data_temporal_validation.csv   (1,822 records)

Table 1